# Dataset summarization extraction

This notebook scans every `.docx` and `.pdf` file under `dataset/`, extracts the requested fields, and writes the final CSV to `dataset_summaries.csv`.

In [ ]:
import re
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET

import pandas as pd

try:
    from pypdf import PdfReader
except Exception:
    PdfReader = None

try:
    import fitz
except Exception:
    fitz = None

DATASET_DIR = Path("dataset")
OUTPUT_CSV = Path("dataset_summaries.csv")
NS = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
MONTH_REGEX = r"(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}"

DOMAIN_KEYWORDS = [
    "Development",
    "Engineering",
    "Testing",
    "Design",
    "Security",
    "Analysis",
    "DevOps",
    "Platform",
    "UI/UX",
]

KEYWORD_SKILLS = {
    "react": "React",
    "typescript": "TypeScript",
    "tailwind": "TailwindCSS",
    "storybook": "Storybook",
    "lighthouse": "Lighthouse",
    "playwright": "Playwright",
    "wcag": "WCAG compliance",
    "aria": "Accessibility",
    "sql": "SQL",
    "graphql": "GraphQL",
    "kafka": "Kafka",
    "redis": "Redis",
    "node.js": "Node.js",
    "docker": "Docker",
    "kubernetes": "Kubernetes",
    "terraform": "Terraform",
    "aws": "AWS",
    "grafana": "Grafana",
    "pagerduty": "PagerDuty",
    "python": "Python",
    "pandas": "pandas",
    "airflow": "Airflow",
    "postgresql": "PostgreSQL",
    "ocr": "OCR",
    "rag": "RAG",
    "ner": "NER",
    "sentiment": "Sentiment analysis",
    "figma": "Figma",
    "design system": "Design systems",
    "usability": "Usability testing",
    "a/b": "A/B test analysis",
    "funnel": "Funnel analysis",
    "forecast": "Forecasting",
    "segmentation": "Customer segmentation",
    "fraud": "Fraud detection",
    "recommendation": "Recommendation systems",
    "xss": "XSS testing",
    "sqli": "SQL injection testing",
    "idor": "IDOR testing",
    "owasp": "OWASP testing",
    "burp": "Burp Suite",
    "nmap": "Nmap",
    "bloodhound": "BloodHound",
    "mimikatz": "Mimikatz",
    "jmeter": "JMeter",
    "load test": "Load testing",
}


def normalize_ws(text):
    text = text.replace("\xa0", " ")
    text = text.replace("–", "-")
    text = text.replace("—", " - ")
    text = re.sub(r"[\u2022\u25aa\u25cf\uf0b7]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_name(value):
    return normalize_ws(value or "").strip(" |:-")


def read_docx_lines(path: Path):
    with ZipFile(path) as zf:
        xml = zf.read("word/document.xml")
    root = ET.fromstring(xml)
    lines = []
    for para in root.findall(".//w:p", NS):
        parts = [node.text for node in para.findall(".//w:t", NS) if node.text]
        line = "".join(parts).strip()
        if line:
            lines.append(clean_name(line))
    return lines


def read_pdf_lines(path: Path):
    if PdfReader is not None:
        try:
            reader = PdfReader(str(path))
            lines = []
            for page in reader.pages:
                page_text = page.extract_text() or ""
                lines.extend([clean_name(line) for line in page_text.splitlines() if clean_name(line)])
            if lines:
                return lines
        except Exception:
            pass
    if fitz is not None:
        doc = fitz.open(path)
        try:
            lines = []
            for page in doc:
                lines.extend([clean_name(line) for line in page.get_text().splitlines() if clean_name(line)])
            return lines
        finally:
            doc.close()
    return []


def read_file_lines(path: Path):
    if path.suffix.lower() == ".docx":
        return read_docx_lines(path)
    if path.suffix.lower() == ".pdf":
        return read_pdf_lines(path)
    return []


def joined_text(lines):
    return normalize_ws(" \\n ".join(lines))


def infer_seniority(role):
    lowered = role.lower()
    for label in ["Senior", "Lead", "Principal", "Staff", "Junior", "Intern"]:
        if label.lower() in lowered:
            return label
    return "Unspecified"


def infer_work_type(title, text):
    blob = f"{title} {text}".lower()
    checks = [
        ("pentest", "Testing / Security Assessment"),
        ("assessment", "Assessment"),
        ("audit", "Audit"),
        ("analysis", "Analysis"),
        ("research", "Research"),
        ("review", "Review"),
        ("migration", "Development / Migration"),
        ("optimization", "Optimization"),
        ("load testing", "Testing"),
        ("stress test", "Testing"),
        ("testing", "Testing"),
        ("postmortem", "Postmortem"),
        ("incident", "Incident Response"),
        ("evaluation", "Evaluation"),
        ("model", "Model Development"),
        ("redesign", "Development"),
        ("progress", "Progress Report"),
    ]
    for keyword, label in checks:
        if keyword in blob:
            return label
    return "Technical Report"


def parse_generic_header(lines):
    title = lines[0] if lines else ""
    author = ""
    team = ""
    date = ""
    if len(lines) > 1 and "|" in lines[1]:
        parts = [clean_name(part) for part in lines[1].split("|") if clean_name(part)]
        remaining = []
        for part in parts:
            if part.lower().startswith("by:"):
                payload = clean_name(part.split(":", 1)[1])
                if re.fullmatch(MONTH_REGEX, payload):
                    date = payload
                else:
                    author = payload
            elif re.fullmatch(MONTH_REGEX, part):
                date = part
            else:
                remaining.append(part)
        for part in remaining:
            if any(keyword.lower() in part.lower() for keyword in DOMAIN_KEYWORDS) and not team:
                team = part
            elif not author:
                author = part
        if not team and len(remaining) >= 2:
            team = remaining[0]
            author = remaining[1]
    if not date:
        for line in lines[:5]:
            match = re.search(MONTH_REGEX, line)
            if match:
                date = match.group(0)
                break
    if not author:
        for line in lines[:5]:
            match = re.search(r"BY:\s*([A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+){1,3})", line, re.IGNORECASE)
            if match:
                author = clean_name(match.group(1))
                break
    return title, author, team, date


def extract_skills(lines, text):
    skills = set()
    for idx, line in enumerate(lines):
        if line.lower().startswith("technologies / tools") or line.lower().startswith("tools used"):
            if idx + 1 < len(lines):
                for piece in re.split(r",|/| and ", lines[idx + 1]):
                    piece = clean_name(piece)
                    if 2 <= len(piece) <= 50:
                        skills.add(piece)
    for keyword, label in KEYWORD_SKILLS.items():
        if keyword in text.lower():
            skills.add(label)
    return "; ".join(sorted(skills))


def extract_time_info(lines, text):
    period = ""
    date = ""
    for idx, line in enumerate(lines):
        if line == "Report Period" and idx + 1 < len(lines):
            period = lines[idx + 1]
        if line == "Date Issued" and idx + 1 < len(lines):
            date = lines[idx + 1]
    if not period:
        match = re.search(r"(Q[1-4]\s+\d{4}[^\n]*)", text)
        if match:
            period = clean_name(match.group(1))
    if not date:
        for line in lines[:6]:
            match = re.search(MONTH_REGEX, line)
            if match:
                date = match.group(0)
                break
    testing = ""
    for line in lines:
        if line.lower().startswith("testing period:"):
            testing = clean_name(line.split(":", 1)[1])
            break
    timeline = " | ".join([value for value in [period, date, testing] if value])
    recency = "Historical"
    if "2026" in timeline:
        recency = "Current"
    elif "2025" in timeline:
        recency = "Recent historical"
    return timeline, period, date, recency


def infer_collaboration(text, department):
    lowered = text.lower()
    if any(term in lowered for term in ["blue team", "cross-team", "frontend dev team", "design/dev sync", "peer reviewed", "stakeholder"]):
        return "Cross-team collaboration mentioned"
    if any(term in lowered for term in ["manager", "rotation", "team", "mentored", "reviewed"]):
        return f"Team-based execution within {department or 'the listed team'}"
    return "Likely individual contribution; no explicit collaboration noted"


def collect_impact(lines):
    picks = []
    for line in lines:
        lower = line.lower()
        if any(token in lower for token in ["improved", "reduced", "found", "identified", "resolved", "achieved", "critical", "high", "impact", "score", "uptime", "mttr", "throughput", "error rate", "completed"]):
            picks.append(line)
        if len(picks) == 3:
            break
    return " | ".join(picks)


def infer_complexity(text, total_hours):
    score = 1 + min(total_hours // 40, 4)
    reasons = []
    if total_hours:
        reasons.append(f"{total_hours}h reported")
    for label, pattern in [
        ("critical findings", r"(\\d+)\\s+Critical"),
        ("high findings", r"(\\d+)\\s+High"),
        ("hosts", r"([\\d,]+)\\s+hosts"),
        ("pages", r"([\\d,]+)\\+?\\s+pages"),
        ("components", r"([\\d,]+)\\s+components"),
        ("incidents", r"(\\d+)\\s+incidents"),
        ("records_millions", r"(\\d+)M\\s+user interaction records"),
        ("users", r"([\\d,]+)\\s+Concurrent Users"),
    ]:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            value = int(match.group(1).replace(",", ""))
            reasons.append(f"{label}: {match.group(1)}")
            score += 2 if value >= 50 else 1
    if re.search(r"99\.99%|50K TPS|domain admin|APT|PCI-DSS", text, re.IGNORECASE):
        score += 2
        reasons.append("high-risk or high-scale scope")
    if re.search(r"blue team|cross-team|stakeholder|peer reviewed", text, re.IGNORECASE):
        score += 1
        reasons.append("collaborative scope")
    score = min(score, 10)
    level = "Low" if score <= 3 else "Medium" if score <= 5 else "High" if score <= 7 else "Very High"
    return level, score, "; ".join(reasons)


def extract_employee_report(path, lines, text, domain):
    data = {"employee_name": "", "employee_role": "", "employee_department": "", "employee_team": ""}
    for idx, line in enumerate(lines):
        if line == "Employee Name" and idx + 1 < len(lines):
            data["employee_name"] = lines[idx + 1]
        elif line == "Role / Title" and idx + 1 < len(lines):
            data["employee_role"] = lines[idx + 1]
        elif line == "Department" and idx + 1 < len(lines):
            data["employee_department"] = lines[idx + 1]
            data["employee_team"] = lines[idx + 1]

    task_titles = []
    for idx, line in enumerate(lines):
        if line == "Description" and idx > 0:
            title = clean_name(re.sub(r"^[^A-Za-z0-9]+", "", lines[idx - 1]))
            if 3 <= len(title) <= 120:
                task_titles.append(title)

    total_hours = sum(int(match) for match in re.findall(r"(\d+)h", text))
    rating_match = re.search(r"Overall Performance Rating:\s*([0-9.]+\s*/\s*[0-9.]+)", text)
    rating = rating_match.group(1) if rating_match else ""
    impact = collect_impact(lines)
    level, score, reason = infer_complexity(text, total_hours)
    timeline, period, date, recency = extract_time_info(lines, text)

    return {
        "employee_name": data["employee_name"],
        "employee_role": data["employee_role"],
        "employee_department": data["employee_department"] or domain,
        "employee_seniority": infer_seniority(data["employee_role"]),
        "employee_team": data["employee_team"] or domain,
        "report_title": "; ".join(task_titles) if task_titles else path.stem,
        "work_type": "Performance Report / Delivery Summary",
        "skills_competency": extract_skills(lines, text),
        "domain_category": domain,
        "outcome_result": f"Completed {len(task_titles)} tasks with {total_hours}h total effort." + (f" Overall rating {rating}." if rating else ""),
        "impact_metrics": impact,
        "complexity_level": level,
        "complexity_score": score,
        "complexity_reason": reason,
        "time_timeline": timeline,
        "report_period": period,
        "issued_or_report_date": date,
        "recency_label": recency,
        "collaboration": infer_collaboration(text, data["employee_department"] or domain),
        "document_type": "employee_performance_report",
        "task_count": len(task_titles),
        "hours_invested_total": total_hours,
        "performance_rating": rating,
    }


def extract_general_report(path, lines, text, domain):
    title, author, team, header_date = parse_generic_header(lines)
    impact = collect_impact(lines)
    timeline, period, date, recency = extract_time_info(lines, text)
    if not date:
        date = header_date
        timeline = " | ".join([value for value in [period, date] if value])
    level, score, reason = infer_complexity(text, 0)

    return {
        "employee_name": author,
        "employee_role": "",
        "employee_department": team or domain,
        "employee_seniority": "Unspecified",
        "employee_team": team or domain,
        "report_title": title or path.stem,
        "work_type": infer_work_type(title or path.stem, text),
        "skills_competency": extract_skills(lines, text),
        "domain_category": domain,
        "outcome_result": impact or "Technical findings and recommendations captured in the report.",
        "impact_metrics": impact,
        "complexity_level": level,
        "complexity_score": score,
        "complexity_reason": reason,
        "time_timeline": timeline,
        "report_period": period,
        "issued_or_report_date": date,
        "recency_label": recency if (recency != "Historical" or date or period) else "Unspecified",
        "collaboration": infer_collaboration(text, team or domain),
        "document_type": "technical_report",
        "task_count": sum(1 for line in lines if any(token in line.lower() for token in ["finding", "recommendation", "metric", "impact", "phase", "components"])),
        "hours_invested_total": 0,
        "performance_rating": "",
    }


rows = []
for path in sorted(DATASET_DIR.rglob("*")):
    if path.suffix.lower() not in {".docx", ".pdf"}:
        continue
    lines = read_file_lines(path)
    text = joined_text(lines)
    domain = path.parent.name
    if "Work Performance Report" in text:
        record = extract_employee_report(path, lines, text, domain)
    else:
        record = extract_general_report(path, lines, text, domain)
    record.update({
        "source_file": path.name,
        "source_path": path.as_posix(),
        "file_extension": path.suffix.lower(),
        "source_folder": path.parent.name,
        "text_length": len(text),
    })
    rows.append(record)

df = pd.DataFrame(rows)
column_order = [
    "source_file",
    "source_path",
    "file_extension",
    "source_folder",
    "employee_name",
    "employee_role",
    "employee_department",
    "employee_seniority",
    "employee_team",
    "report_title",
    "work_type",
    "skills_competency",
    "domain_category",
    "outcome_result",
    "impact_metrics",
    "complexity_level",
    "complexity_score",
    "complexity_reason",
    "time_timeline",
    "report_period",
    "issued_or_report_date",
    "recency_label",
    "collaboration",
    "document_type",
    "task_count",
    "hours_invested_total",
    "performance_rating",
    "text_length",
]
df = df[column_order]
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} rows to {OUTPUT_CSV}")
df.head(10)
